# Algorithm Agent

In [ ]:
%env AWS_PROFILE=kmnlp
%env ZARR_REFERENCE_PATH=s3://data-c6c22a2e42294c11b52ee7f0c792c071/crw/5km/v3.1/nc/v1.0/daily/sst/zarr_reference.json

In [ ]:
import sys
sys.path.append("../src")

In [ ]:
import os
import fsspec
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = cluster.get_client()

zarr_reference_path = os.getenv('ZARR_REFERENCE_PATH')
mapper = fsspec.get_mapper(
    'reference://', 
    fo=zarr_reference_path,
    remote_protocol='s3',
)
ds = xr.open_zarr(mapper, consolidated=False)
sst = ds.analysed_sst
sst

---

In [ ]:
from rich import print as rprint
from e84_geoai_common.llm.models import BedrockClaudeLLM, CLAUDE_BEDROCK_MODEL_IDS

from llm_agent.agents.algorithm_agent.algorithm.base import *
from llm_agent.agents.algorithm_agent.algorithm.data_array_functions import *
from llm_agent.agents.algorithm_agent.algorithm.general_plot_functions import *
from llm_agent.agents.algorithm_agent.algorithm.exec import *
from llm_agent.agents.algorithm_agent.algorithm_agent import *
from llm_agent.agents.algorithm_agent.interface import *

In [ ]:
input_vars = [
    InputVariable(
        name="sst",
        variable_type="DataArray",
        description="""\
The SST (Sea Surface Temperature) variable is a 3-dimensional array of data representing global sea surface temperatures. The dimensions of this array are:

Time: Covering a period from January 1, 2020, to November 30, 2024, with daily measurements.
Latitude: Ranging from approximately -90° (South Pole) to +90° (North Pole), with 3600 evenly spaced points.
Longitude: Ranging from -180° to +180°, covering the entire globe with 7200 evenly spaced points.

Each point in this 3D array represents the sea surface temperature at a specific location (defined by latitude and longitude) and time. The temperatures are measured in degrees Celsius, with valid values ranging from -20°C to +50°C.

This dataset provides a comprehensive, high-resolution (5km) global view of sea surface temperatures over time. It's designed for use in various oceanographic and climate studies, particularly for monitoring coral reef environments as indicated by its association with the NOAA Coral Reef Watch program.\
"""
    )
]

---

In [ ]:
llm = BedrockClaudeLLM(model_id=CLAUDE_BEDROCK_MODEL_IDS['Claude 3.5 Sonnet'])

In [ ]:
agent = AlgorithmAgent(
    llm, 
    problem_spec=AlgorithmProblemSpec, 
    solution_spec=AlgorithmSolutionSpec[DataArrayFunctionSet | GeneralPlottingFunctionSet],
)

In [ ]:
rprint(agent.system_prompt)

---

## Problem 1

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="What was the mean SST in the Black Sea on June 01, 2024.",
    inputs_provided=input_vars,
    outputs_requested=[OutputVariable(name="sst_mean_scalar", type_as_str="float", description="The mean value of the SST.")],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.algorithm)
rprint(solution.explanation)

Python equivalent:

```python
# Polygon representation of the Black Sea
black_sea_polygon = GeocodeNaturalLanguageDescription(text='Black Sea')

# SST data for June 01, 2024
sst_june_01_2024 = FilterDataArray(data_array=sst, time_range=('2024-06-01', '2024-06-01'))

# SST data for the Black Sea on June 01, 2024
sst_black_sea = FilterDataArrayByPolygon(data_array=sst_june_01_2024, polygon=black_sea_polygon)

# Mean SST for the Black Sea on June 01, 2024
sst_mean = ReduceDataArray(data_array=sst_black_sea, op='mean')

# Mean SST for the Black Sea on June 01, 2024 as a scalar value
sst_mean_scalar = DataArrayToScalar(data_array=sst_mean)

outputs = {
    "sst_mean_scalar": sst_mean_scalar,
}
```

---

### Execute

In [ ]:
execute_algorithm(solution.algorithm, input_vars={"sst": sst})

---
---

## Problem 2

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="Show the daily average SST within the Chesapeake Bay over the the months Jan 2020 and Jan 2024 as two separate time series on the same plot. The x-axis should be agnostic of the year so that the two time series are on top of each other.",
    inputs_provided=input_vars,
    outputs_requested=[OutputVariable(name="time_series_plot", type_as_str="plot", description="The time series plot.")],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
execute_algorithm(solution.algorithm, input_vars={"sst": sst})

---

## Problem 3

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="Draw a bar chart comparing the max SST's in Chesapeake Bay, Delware Bay, and San Pablo Bay in the month of June 2024.",
    inputs_provided=input_vars,
    outputs_requested=[OutputVariable(name="bar_plot", type_as_str="plot", description="The bar plot.")],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
execute_algorithm(solution.algorithm, input_vars={"sst": sst})

---

## Problem 4

In [ ]:
problem_spec = AlgorithmProblemSpec(
    statement="Plot the SST maps for Jan 1, 2023 and Aug 1, 2023 side by side.",
    inputs_provided=input_vars,
    outputs_requested=[OutputVariable(name="plot", type_as_str="plot", description="The requested plot.")],
)
rprint(problem_spec)

In [ ]:
solution = agent.run(problem_spec)

In [ ]:
rprint(solution.thoughts)
rprint(solution.algorithm)
rprint(solution.explanation)

---

### Execute

In [ ]:
execute_algorithm(solution.algorithm, input_vars={"sst": sst})